In [1]:
import os
from dotenv import load_dotenv
import json

In [2]:
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown
from google.api_core import retry
def to_markdown(text):
    text = text.replace('•', '  *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [3]:
load_dotenv()
google_api_key = os.getenv('GEMINI_API_KEY')

In [4]:
genai.configure(api_key=google_api_key)

In [5]:
system_prompt ={"text":"you are an medical AI assistant,you should predict the reaction between the medicine and effect on body."}

In [6]:
# tool to add medicine names to a list
medicine_list = []
def medicine_tool(*medicine):
    med = medicine.lower().strip()
    if med not in medicine_list:
        medicine_list.append(med)
    return medicine_list
    

In [7]:
medicine_function={
    "name": "medicine_tool",
    "description":"gets the list of medicines.Call this whenever you want to get the list of medicines, for example, when the user asks for a list of medicines, you can call this function to get the list of medicines.",
    "parameters":{
        "type":"object",
        "properties":{
            "medicine":{
                "type":"string",
                "description":"The medicine name",
            },
        },
        "required":["medicine"],
        "additionalProperties":False
    }
}

In [8]:
tool_config = {
    "function_calling_config": {
        "mode": "ANY",
        "allowed_function_names": ["medicine_function"]
    }
}


In [9]:
user_prompt=[medicine_list]

In [10]:
def message_gemini(*user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    gemini = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_prompt,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

In [12]:
to_markdown(message_gemini("paracetamol","ibuprofen","aspirin"))

> I cannot provide specific medical advice or predict the exact reactions of combining paracetamol (acetaminophen), ibuprofen, and aspirin.  This is because:
> 
> * **Drug Interactions:** Combining these three medications can lead to serious interactions and increased risk of side effects, especially liver and kidney damage.  Aspirin, ibuprofen, and paracetamol all have the potential to cause gastrointestinal bleeding, and taking them together significantly increases this risk.  Liver damage is a particular concern when combining paracetamol with other drugs that also affect the liver.
> 
> * **Individual Variability:** Reactions to medications vary greatly depending on factors such as age, weight, overall health, other medications being taken, and genetic predispositions.
> 
> * **Dosage:** The dosage of each medication significantly impacts potential side effects and interactions.
> 
> **If you are considering taking these medications together, it is absolutely crucial to consult a doctor or pharmacist.** They can assess your individual health status, consider potential interactions, and determine if this combination is safe and appropriate for you.  They can also advise on appropriate dosages to minimize risk.
> 
> **Do not self-treat.** Taking these medications together without professional guidance could have severe consequences.  Always seek medical advice before mixing medications.
